# Klasifikasi Gambar Bunga — MobileNetV2 (Deep Learning)

**Mata Kuliah:** Machine Learning (UAS — Metode Deep Learning)  
**Dataset:** Bunga Melati Jakarta, Melati Jepang, Bintaro, dan Tapak Dara  
**Total Data:** 1.440 gambar (4 kelas × 360 gambar)  

---
**Referensi Jurnal:**  
Koklu, M., Unlersen, M. F., Ozkan, I. A., Aslan, M. F., & Sabanci, K. (2022).  
A CNN-SVM study based on selected deep features for grapevine leaves classification.  
*Measurement*, 188, 110425. https://doi.org/10.1016/j.measurement.2021.110425

---
**Arsitektur:**  
MobileNetV2 pretrained ImageNet → Fine-Tuning (2 fase) → Klasifikasi 4 kelas bunga

**Alur:**
1. Import Library
2. Konfigurasi & Struktur Dataset
3. Load & Preprocessing Data
4. Data Augmentation
5. Arsitektur MobileNetV2 (Transfer Learning)
6. Training Phase 1 — Feature Extraction (Base Frozen)
7. Training Phase 2 — Fine-Tuning (Unfreeze 50 Layer Terakhir)
8. Evaluasi Model
9. Visualisasi Hasil
10. Demo Prediksi Input Baru
11. Simpan Model & Ringkasan

## 1. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
import time

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D,
    Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Evaluasi
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    matthews_corrcoef
)

print(f"✅ Semua library berhasil diimport")
print(f"   TensorFlow version : {tf.__version__}")
print(f"   GPU tersedia       : {tf.config.list_physical_devices('GPU')}")

## 2. Konfigurasi

> **Sesuaikan `DATASET_PATH` dengan lokasi folder dataset kamu.**  
> Struktur folder yang diharapkan:
> ```
> dataset/
> ├── melati_jakarta/   (360 gambar)
> ├── melati_jepang/    (360 gambar)
> ├── bintaro/          (360 gambar)
> └── tapak_dara/       (360 gambar)
> ```

In [ ]:
# ============================================================
#  SESUAIKAN PATH DATASET
# ============================================================
DATASET_PATH = './dataset'   # Ganti jika lokasi berbeda

CLASS_NAMES  = ['melati_jakarta', 'melati_jepang', 'bintaro', 'tapak_dara']
NUM_CLASSES  = len(CLASS_NAMES)

# ── Parameter gambar (Koklu et al., 2022) ───────────────────
IMG_SIZE     = 224       # Input size standar MobileNetV2

# ── Parameter training ──────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS_P1    = 20        # Phase 1: base frozen
EPOCHS_P2    = 10        # Phase 2: fine-tuning
LR_P1        = 0.001     # Learning rate phase 1
LR_P2        = 0.0001    # Learning rate phase 2 (lebih kecil)
UNFREEZE_N   = 50        # Jumlah layer terakhir yang di-unfreeze

# ── Split data ───────────────────────────────────────────────
TEST_SIZE    = 0.20      # 20% test
VAL_SIZE     = 0.10      # 10% validasi
RANDOM_STATE = 42

print(f"📁 Dataset path    : {DATASET_PATH}")
print(f"🌸 Kelas           : {CLASS_NAMES}")
print(f"📐 Ukuran gambar   : {IMG_SIZE}×{IMG_SIZE}")
print(f"🔢 Batch size      : {BATCH_SIZE}")
print(f"🔄 Epochs Phase 1  : {EPOCHS_P1}")
print(f"🔄 Epochs Phase 2  : {EPOCHS_P2}")
print(f"📊 Split           : Train 70% | Val 10% | Test 20%")

## 3. Eksplorasi & Load Dataset

In [ ]:
# ── Hitung jumlah gambar per kelas ───────────────────────────
class_counts = {}
for cls in CLASS_NAMES:
    path = os.path.join(DATASET_PATH, cls)
    count = len([f for f in os.listdir(path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))])
    class_counts[cls] = count

print("📊 Distribusi Dataset:")
print("-" * 35)
for cls, cnt in class_counts.items():
    print(f"  {cls:<20} : {cnt} gambar")
print("-" * 35)
print(f"  {'TOTAL':<20} : {sum(class_counts.values())} gambar")

In [ ]:
# ── Visualisasi distribusi kelas ─────────────────────────────
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']
labels_display = [c.replace('_', ' ').title() for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(labels_display, class_counts.values(),
                   color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Distribusi Data per Kelas', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Kelas Bunga')
axes[0].set_ylabel('Jumlah Gambar')
for bar, val in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', va='bottom', fontweight='bold')

axes[1].pie(class_counts.values(), labels=labels_display,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporsi Kelas Dataset', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('01_distribusi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 01_distribusi_dataset.png")

In [ ]:
# ── Load semua gambar ─────────────────────────────────────────
def load_dataset(dataset_path, class_names, img_size):
    """
    Load gambar dari folder, resize ke img_size × img_size.
    Returns:
        X : np.array shape (N, img_size, img_size, 3) — nilai uint8 [0,255]
        y : np.array shape (N,) — label integer
    """
    X, y = [], []
    for label, cls in enumerate(class_names):
        cls_path = os.path.join(dataset_path, cls)
        images = [f for f in os.listdir(cls_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
        print(f"  [{cls}] → {len(images)} gambar")
        for img_file in images:
            try:
                img = Image.open(os.path.join(cls_path, img_file)).convert('RGB')
                img = img.resize((img_size, img_size))
                X.append(np.array(img))
                y.append(label)
            except Exception as e:
                print(f"  ⚠️  Gagal load {img_file}: {e}")
    return np.array(X), np.array(y)


print("⏳ Memuat dataset ...")
X, y = load_dataset(DATASET_PATH, CLASS_NAMES, IMG_SIZE)

print(f"\n✅ Dataset dimuat")
print(f"   Shape X : {X.shape}  (N, H, W, C)")
print(f"   Shape y : {y.shape}")
print(f"   Dist    : {dict(zip(CLASS_NAMES, [np.sum(y==i) for i in range(NUM_CLASSES)]))}")

In [ ]:
# ── Contoh gambar per kelas ───────────────────────────────────
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Contoh Gambar Dataset per Kelas Bunga', fontsize=15, fontweight='bold')

for row_idx, (cls, label) in enumerate(zip(CLASS_NAMES, range(NUM_CLASSES))):
    idxs = np.where(y == label)[0]
    samples = np.random.choice(idxs, 5, replace=False)
    for col_idx, idx in enumerate(samples):
        axes[row_idx, col_idx].imshow(X[idx])
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(
        cls.replace('_', ' ').title(),
        fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('02_contoh_gambar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 02_contoh_gambar.png")

## 4. Preprocessing & Pembagian Data

Mengikuti Koklu et al. (2022):
- Preprocessing MobileNetV2: normalisasi piksel ke rentang **[-1, 1]** via `preprocess_input()`
- One-hot encoding label
- Split **70% train | 10% validasi | 20% test** dengan stratified split

In [ ]:
# ── Preprocessing MobileNetV2 (normalisasi [-1, 1]) ──────────
X_norm = preprocess_input(X.astype(np.float32))
y_onehot = to_categorical(y, NUM_CLASSES)

# ── Split: Train+Val (80%) vs Test (20%) ─────────────────────
X_trainval, X_test, y_trainval_oh, y_test_oh, y_trainval, y_test = train_test_split(
    X_norm, y_onehot, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# ── Split: Train (87.5% dari trainval) vs Val (12.5%) ────────
X_train, X_val, y_train_oh, y_val_oh, y_train, y_val = train_test_split(
    X_trainval, y_trainval_oh, y_trainval,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=y_trainval
)

total = len(X_norm)
print("📊 Pembagian Dataset:")
print(f"   Training   : {len(X_train):4d} sampel ({len(X_train)/total*100:.1f}%)")
print(f"   Validasi   : {len(X_val):4d} sampel ({len(X_val)/total*100:.1f}%)")
print(f"   Testing    : {len(X_test):4d} sampel ({len(X_test)/total*100:.1f}%)")
print(f"   Total      : {total} sampel")

## 5. Data Augmentation

Mengikuti Koklu et al. (2022) Tabel 2 — augmentasi diterapkan **hanya pada data training**:
- Refleksi horizontal (`horizontal_flip`)
- Rotasi ±45° (`rotation_range=45`)
- Scaling 80%–120% (`zoom_range=0.2`)
- Translasi horizontal & vertikal (`width/height_shift_range=0.1`)
- Variasi kecerahan (`brightness_range=[0.8, 1.2]`)

In [ ]:
# ── Definisi augmentasi ───────────────────────────────────────
train_datagen = ImageDataGenerator(
    horizontal_flip=True,          # Refleksi (Koklu et al., 2022)
    rotation_range=45,             # Rotasi ±45° (Koklu et al., 2022)
    zoom_range=0.2,                # Scaling 80–120% (Koklu et al., 2022)
    width_shift_range=0.1,         # Translasi horizontal
    height_shift_range=0.1,        # Translasi vertikal
    brightness_range=[0.8, 1.2],   # Variasi kecerahan
    fill_mode='nearest'
)
val_datagen  = ImageDataGenerator()   # Tanpa augmentasi

# ── Generator ─────────────────────────────────────────────────
train_gen = train_datagen.flow(
    X_train, y_train_oh,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_gen = val_datagen.flow(
    X_val, y_val_oh,
    batch_size=BATCH_SIZE,
    shuffle=False
)

steps_per_epoch = len(X_train) // BATCH_SIZE

print("✅ Data augmentation dikonfigurasi")
print(f"   Steps per epoch : {steps_per_epoch}")

In [ ]:
# ── Visualisasi efek augmentasi ───────────────────────────────
sample_img = X[0][np.newaxis, ...]  # Ambil 1 gambar
aug_gen = train_datagen.flow(sample_img, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Efek Data Augmentation — Koklu et al. (2022)',
             fontsize=14, fontweight='bold')

# Gambar asli
axes[0, 0].imshow(X[0])
axes[0, 0].set_title('Asli', fontsize=9, fontweight='bold')
axes[0, 0].axis('off')

# Gambar augmentasi
for idx in range(1, 10):
    aug_img = next(aug_gen)[0]
    # Denormalisasi dari [-1,1] ke [0,1] untuk display
    aug_img_disp = np.clip((aug_img + 1) / 2, 0, 1)
    row, col = idx // 5, idx % 5
    axes[row, col].imshow(aug_img_disp)
    axes[row, col].set_title(f'Augmented {idx}', fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('03_augmentasi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 03_augmentasi.png")

## 6. Arsitektur MobileNetV2 (Transfer Learning)

Arsitektur MobileNetV2 menggunakan *depthwise separable convolution* yang menjadikannya jauh lebih ringan dibandingkan VGG16 namun tetap kompetitif dalam akurasi. Dalam penelitian ini digunakan pendekatan transfer learning dua fase mengikuti Koklu et al. (2022):

**Phase 1** — Base model di-*freeze*, hanya layer tambahan (classifier head) yang dilatih.  
**Phase 2** — Fine-tuning: 50 layer terakhir base model dibuka (*unfreeze*) dan dilatih dengan learning rate lebih kecil.

In [ ]:
# ── Load MobileNetV2 pretrained ImageNet ──────────────────────
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,              # Hapus FC layer ImageNet
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze seluruh base model (Phase 1)
base_model.trainable = False

# ── Tambah classifier head ────────────────────────────────────
x = base_model.output
x = GlobalAveragePooling2D(name='gap')(x)          # Reduksi spasial
x = BatchNormalization(name='bn_head')(x)           # Normalisasi batch
x = Dense(256, activation='relu', name='fc_256')(x)
x = Dropout(0.5, name='drop_1')(x)
x = Dense(128, activation='relu', name='fc_128')(x)
x = Dropout(0.3, name='drop_2')(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name='output')(x)

mobilenet_model = Model(inputs=base_model.input, outputs=outputs)

# Hitung parameter
total_params     = mobilenet_model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in mobilenet_model.trainable_weights])

print("✅ Arsitektur MobileNetV2 siap")
print(f"   Total parameter     : {total_params:,}")
print(f"   Parameter dilatih   : {trainable_params:,} (Phase 1)")
print(f"   Parameter di-freeze : {total_params - trainable_params:,}")
print("\n📐 Classifier Head:")
print("   MobileNetV2 Base → GlobalAveragePooling2D → BatchNorm")
print("   → Dense(256, ReLU) → Dropout(0.5)")
print("   → Dense(128, ReLU) → Dropout(0.3)")
print("   → Dense(4, Softmax)")

## 7. Training Phase 1 — Feature Extraction (Base Frozen)

In [ ]:
# ── Kompilasi Phase 1 ─────────────────────────────────────────
mobilenet_model.compile(
    optimizer=Adam(learning_rate=LR_P1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────
callbacks_p1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'mobilenetv2_phase1_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print(f"⏳ Training Phase 1 (base frozen, lr={LR_P1}) ...")
start = time.time()

history1 = mobilenet_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_P1,
    validation_data=val_gen,
    callbacks=callbacks_p1,
    verbose=1
)

elapsed1 = time.time() - start
best_val_p1 = max(history1.history['val_accuracy'])
print(f"\n✅ Phase 1 selesai dalam {elapsed1/60:.1f} menit")
print(f"   Best Val Accuracy Phase 1 : {best_val_p1*100:.2f}%")

## 8. Training Phase 2 — Fine-Tuning (Unfreeze 50 Layer Terakhir)

In [ ]:
# ── Unfreeze 50 layer terakhir ────────────────────────────────
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_N]:
    layer.trainable = False

trainable_now = sum([tf.size(w).numpy() for w in mobilenet_model.trainable_weights])
print(f"🔓 Fine-tuning: {UNFREEZE_N} layer terakhir dibuka")
print(f"   Parameter dilatih sekarang : {trainable_now:,}")

# ── Kompilasi ulang dengan lr lebih kecil ────────────────────
mobilenet_model.compile(
    optimizer=Adam(learning_rate=LR_P2),   # lr lebih kecil 10x
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        'mobilenetv2_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print(f"\n⏳ Training Phase 2 (fine-tuning, lr={LR_P2}) ...")
start = time.time()

history2 = mobilenet_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_P2,
    validation_data=val_gen,
    callbacks=callbacks_p2,
    verbose=1
)

elapsed2 = time.time() - start
best_val_p2 = max(history2.history['val_accuracy'])
print(f"\n✅ Phase 2 selesai dalam {elapsed2/60:.1f} menit")
print(f"   Best Val Accuracy Phase 2 : {best_val_p2*100:.2f}%")

## 9. Visualisasi Kurva Training

In [ ]:
# ── Gabungkan history Phase 1 + Phase 2 ──────────────────────
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined

all_hist = combine_histories(history1, history2)
ep_total = len(all_hist['accuracy'])
ep_p1    = len(history1.history['accuracy'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Kurva Training MobileNetV2 (Phase 1 + Fine-Tuning)',
             fontsize=14, fontweight='bold')

epoch_range = range(1, ep_total + 1)

plot_cfg = [
    ('accuracy',     'val_accuracy', 'Training Accuracy vs Epochs',    'Accuracy (%)', '#2196F3', '#4CAF50'),
    ('loss',         'val_loss',     'Training Loss vs Epochs',        'Loss',         '#F44336', '#FF9800'),
    ('accuracy',     'val_accuracy', 'Validation Accuracy vs Epochs',  'Accuracy (%)', '#2196F3', '#4CAF50'),
    ('loss',         'val_loss',     'Validation Loss vs Epochs',      'Loss',         '#F44336', '#FF9800'),
]
train_keys = ['accuracy', 'loss', 'val_accuracy', 'val_loss']
val_keys   = ['val_accuracy', 'val_loss', 'val_accuracy', 'val_loss']
titles     = ['Training Accuracy vs Epochs', 'Training Loss vs Epochs',
               'Validation Accuracy vs Epochs', 'Validation Loss vs Epochs']
ylabels    = ['Accuracy', 'Loss', 'Accuracy', 'Loss']

for ax, train_k, val_k, title, ylabel in zip(
        axes.flat, train_keys[:2] + ['val_accuracy', 'val_loss'],
        val_keys, titles, ylabels):
    pass  # placeholder — pakai plot manual di bawah

# Plot manual lebih bersih
for ax in axes.flat:
    ax.cla()

# Accuracy
axes[0,0].plot(epoch_range, all_hist['accuracy'],     color='#2196F3', lw=2, label='Train')
axes[0,0].plot(epoch_range, all_hist['val_accuracy'], color='#4CAF50', lw=2, ls='--', label='Validasi')
axes[0,0].axvline(ep_p1, color='red', ls=':', lw=1.5, label='Fine-tuning mulai')
axes[0,0].set_title('Training & Validation Accuracy', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Loss
axes[0,1].plot(epoch_range, all_hist['loss'],     color='#F44336', lw=2, label='Train')
axes[0,1].plot(epoch_range, all_hist['val_loss'], color='#FF9800', lw=2, ls='--', label='Validasi')
axes[0,1].axvline(ep_p1, color='red', ls=':', lw=1.5, label='Fine-tuning mulai')
axes[0,1].set_title('Training & Validation Loss', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Hanya Phase 1
ep_p1_range = range(1, ep_p1 + 1)
axes[1,0].plot(ep_p1_range, history1.history['accuracy'],     color='#2196F3', lw=2, label='Train')
axes[1,0].plot(ep_p1_range, history1.history['val_accuracy'], color='#4CAF50', lw=2, ls='--', label='Validasi')
axes[1,0].set_title('Phase 1 — Feature Extraction Accuracy', fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# Hanya Phase 2
ep_p2 = len(history2.history['accuracy'])
ep_p2_range = range(1, ep_p2 + 1)
axes[1,1].plot(ep_p2_range, history2.history['accuracy'],     color='#9C27B0', lw=2, label='Train')
axes[1,1].plot(ep_p2_range, history2.history['val_accuracy'], color='#E91E63', lw=2, ls='--', label='Validasi')
axes[1,1].set_title('Phase 2 — Fine-Tuning Accuracy', fontweight='bold')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('04_kurva_training.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 04_kurva_training.png")

## 10. Evaluasi Model pada Data Testing

In [ ]:
# ── Prediksi pada test set ────────────────────────────────────
print("⏳ Memprediksi data test ...")
y_pred_prob = mobilenet_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)   # Prediksi kelas

# ── Hitung metrik ─────────────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')
mcc  = matthews_corrcoef(y_test, y_pred)

print("\n" + "=" * 50)
print("   HASIL EVALUASI — MobileNetV2 (Fine-Tuning)")
print("=" * 50)
print(f"   Accuracy   : {acc*100:.2f}%")
print(f"   Precision  : {prec:.4f}")
print(f"   Recall     : {rec:.4f}")
print(f"   F1-Score   : {f1:.4f}")
print(f"   MCC        : {mcc:.4f}")
print("=" * 50)

In [ ]:
# ── Classification Report ─────────────────────────────────────
print("\n📋 Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=labels_display
))

## 11. Visualisasi Hasil Evaluasi

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=labels_display,
    yticklabels=labels_display,
    linewidths=0.5, ax=ax
)
ax.set_title(f'Confusion Matrix — MobileNetV2\nAccuracy: {acc*100:.2f}%',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Aktual', fontsize=11)
ax.set_xlabel('Prediksi', fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('05_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 05_confusion_matrix.png")

In [ ]:
# ── Metrik per kelas ─────────────────────────────────────────
prec_cls = precision_score(y_test, y_pred, average=None)
rec_cls  = recall_score(y_test, y_pred, average=None)
f1_cls   = f1_score(y_test, y_pred, average=None)

x = np.arange(NUM_CLASSES)
w = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - w, prec_cls * 100, w, label='Precision', color='#4CAF50', edgecolor='black', alpha=0.85)
ax.bar(x,     rec_cls  * 100, w, label='Recall',    color='#2196F3', edgecolor='black', alpha=0.85)
ax.bar(x + w, f1_cls   * 100, w, label='F1-Score',  color='#FF9800', edgecolor='black', alpha=0.85)
ax.set_title('Metrik Evaluasi per Kelas — MobileNetV2', fontsize=13, fontweight='bold')
ax.set_xlabel('Kelas Bunga')
ax.set_ylabel('Nilai (%)')
ax.set_xticks(x)
ax.set_xticklabels(labels_display, rotation=15)
ax.set_ylim(0, 115)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('06_metrik_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 06_metrik_per_kelas.png")

## 12. Demo Prediksi Input Baru

In [ ]:
# ── Visualisasi 8 sampel prediksi dari test set ───────────────
sample_idx = np.random.choice(len(X_test), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Demo Prediksi — MobileNetV2 (Test Set)',
             fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flat, sample_idx):
    # Denormalisasi [-1,1] → [0,1] untuk display
    img_disp = np.clip((X_test[idx] + 1) / 2, 0, 1)
    true_lbl = labels_display[y_test[idx]]
    pred_lbl = labels_display[y_pred[idx]]
    conf     = y_pred_prob[idx][y_pred[idx]] * 100
    correct  = y_test[idx] == y_pred[idx]

    ax.imshow(img_disp)
    ax.axis('off')
    ax.set_title(
        f'Aktual : {true_lbl}\nPrediksi: {pred_lbl}\n'
        f'Conf: {conf:.1f}%  {"✓" if correct else "✗"}',
        fontsize=8,
        color='green' if correct else 'red',
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('07_demo_prediksi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan: 07_demo_prediksi.png")

In [ ]:
# ── Fungsi prediksi untuk 1 gambar baru ──────────────────────
def predict_flower(image_path, model, class_names, img_size=224):
    """
    Prediksi kelas bunga dari satu gambar.
    Parameters
    ----------
    image_path : str  — path ke file gambar
    model      : trained Keras model
    class_names: list nama kelas
    """
    img     = Image.open(image_path).convert('RGB').resize((img_size, img_size))
    arr     = preprocess_input(np.array(img).astype(np.float32))
    arr_exp = np.expand_dims(arr, axis=0)

    proba      = model.predict(arr_exp, verbose=0)[0]
    pred_idx   = np.argmax(proba)
    pred_label = class_names[pred_idx]

    label_disp = [c.replace('_', ' ').title() for c in class_names]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(img);  axes[0].axis('off')
    axes[0].set_title('Input Gambar', fontsize=11)

    bar_colors = ['#E91E63' if i == pred_idx else '#B0BEC5'
                  for i in range(len(class_names))]
    axes[1].barh(label_disp, proba * 100, color=bar_colors,
                 edgecolor='black', alpha=0.85)
    for i, (val, lbl) in enumerate(zip(proba * 100, label_disp)):
        axes[1].text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_xlim(0, 115)
    axes[1].set_title(
        f'Prediksi: {pred_label.replace("_", " ").title()}\n'
        f'Confidence: {proba[pred_idx]*100:.1f}%',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('08_prediksi_baru.png', dpi=150, bbox_inches='tight')
    plt.show()
    return pred_label, proba[pred_idx]


# ── Contoh penggunaan ─────────────────────────────────────────
# GAMBAR_BARU = 'dataset/tapak_dara/img_001.jpg'   # ← ganti path
# hasil, conf = predict_flower(GAMBAR_BARU, mobilenet_model, CLASS_NAMES)
# print(f'Prediksi: {hasil} ({conf*100:.2f}%)')

print("✅ Fungsi predict_flower siap.")
print("   Hapus komentar (#) pada GAMBAR_BARU untuk mencoba.")

## 13. Simpan Model & Ringkasan

In [ ]:
# ── Simpan model ──────────────────────────────────────────────
mobilenet_model.save('model_mobilenetv2.h5')

# ── Simpan hasil metrik ke CSV ────────────────────────────────
results = pd.DataFrame([{
    'Model'       : 'MobileNetV2 Fine-Tuning (Deep Learning)',
    'Accuracy (%)': round(acc  * 100, 2),
    'Precision'   : round(prec, 4),
    'Recall'      : round(rec,  4),
    'F1-Score'    : round(f1,   4),
    'MCC'         : round(mcc,  4)
}])
results.to_csv('hasil_mobilenetv2.csv', index=False)

print("✅ File tersimpan:")
print("   → model_mobilenetv2.h5")
print("   → hasil_mobilenetv2.csv")

# ── Ringkasan akhir ───────────────────────────────────────────
print("\n" + "=" * 58)
print("   RINGKASAN HASIL — MobileNetV2 (Koklu et al., 2022)")
print("=" * 58)
print(f"   Referensi : Koklu et al. (2022), Measurement, 188")
print(f"   Backbone  : MobileNetV2 (pretrained ImageNet)")
print(f"   Dataset   : 4 kelas × 360 gambar = 1.440 total")
print(f"   Split     : Train 70% | Val 10% | Test 20%")
print(f"   Augmentasi: Flip, Rotasi±45°, Zoom 80-120%, Shift")
print(f"   Phase 1   : Base frozen, lr={LR_P1}")
print(f"   Phase 2   : Fine-tune {UNFREEZE_N} layer, lr={LR_P2}")
print()
print(f"   Accuracy  : {acc*100:.2f}%")
print(f"   Precision : {prec:.4f}")
print(f"   Recall    : {rec:.4f}")
print(f"   F1-Score  : {f1:.4f}")
print(f"   MCC       : {mcc:.4f}")
print()
print("   Output Files:")
files = [
    '01_distribusi_dataset.png',
    '02_contoh_gambar.png',
    '03_augmentasi.png',
    '04_kurva_training.png',
    '05_confusion_matrix.png',
    '06_metrik_per_kelas.png',
    '07_demo_prediksi.png',
    '08_prediksi_baru.png',
    'model_mobilenetv2.h5',
    'hasil_mobilenetv2.csv'
]
for f in files:
    print(f"   → {f}")
print("=" * 58)